In [ ]:
!pip install scikit-surprise


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise import accuracy
import joblib
from collections import defaultdict
from google.colab import drive
import os

In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/Models/'

# Create base directory if it doesn't exist
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
    print(f"Created base directory at {base_dir}")
else:
    print(f"Base directory already exists at {base_dir}")

experiment_name = 'svd_all'
experiment_dir = os.path.join(base_dir, experiment_name)

if not os.path.exists(experiment_dir):
    os.makedirs(experiment_dir)
    print(f"Created experiment directory at {experiment_dir}")
else:
    print(f"Experiment directory already exists at {experiment_dir}")

In [ ]:
df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/all.csv')
df = df.drop('techniques', axis=1)

In [ ]:
df['AuthorId'] = df['AuthorId'].astype(int)
df['RecipeId'] = df['RecipeId'].astype(int)
df['Rating'] = df['Rating'].astype(float)

In [ ]:
reader = Reader(rating_scale=(df['Rating'].min(), df['Rating'].max()))
data = Dataset.load_from_df(df[['AuthorId', 'RecipeId', 'Rating']], reader)


In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
svd_model = SVD(random_state=42,n_factors=100, n_epochs=30)
svd_model.fit(trainset)

In [ ]:
predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)



In [ ]:
cv_results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Average RMSE: {np.mean(cv_results['test_rmse']):.4f}")
print(f"Average MAE: {np.mean(cv_results['test_mae']):.4f}")

In [ ]:
final_model_save_path = os.path.join(experiment_dir, 'svd_model.joblib')

joblib.dump(svd_model, final_model_save_path)
print(f"SVD model saved to {final_model_save_path}")

In [ ]:
breakfast_recipe_ids = df.loc[df["IsBreakfast"] == 1, "RecipeId"].unique()
breakfast_recipe_ids_set = set(breakfast_recipe_ids)

lunch_recipe_ids = df.loc[df["IsLunch"] == 1, "RecipeId"].unique()
lunch_recipe_ids_set = set(lunch_recipe_ids)

snack_recipe_ids = df.loc[df["IsSnack"] == 1, "RecipeId"].unique()
snack_recipe_ids_set = set(snack_recipe_ids)

dinner_recipe_ids = df.loc[df["IsDinner"] == 1, "RecipeId"].unique()
dinner_recipe_ids_set = set(dinner_recipe_ids)

print(len(breakfast_recipe_ids_set))
print(len(lunch_recipe_ids_set))
print(len(snack_recipe_ids_set))
print(len(dinner_recipe_ids_set))

In [ ]:
relevant_breakfast_dict = {}

relevant_lunch_dict = {}

relevant_snack_dict = {}

relevant_dinner_dict = {}

for a, b, c in testset:
    if c > 0.8:
        if b in breakfast_recipe_ids_set:
          if a not in relevant_breakfast_dict:
              relevant_breakfast_dict[a] = []
          relevant_breakfast_dict[a].append(b)
        if b in lunch_recipe_ids_set:
          if a not in relevant_lunch_dict:
              relevant_lunch_dict[a] = []
          relevant_lunch_dict[a].append(b)
        if b in snack_recipe_ids_set:
          if a not in relevant_snack_dict:
              relevant_snack_dict[a] = []
          relevant_snack_dict[a].append(b)
        if b in dinner_recipe_ids_set:
          if a not in relevant_dinner_dict:
              relevant_dinner_dict[a] = []
          relevant_dinner_dict[a].append(b)

In [ ]:
def precision_at_k(recommended_items, relevant_items, k):
    top_k = recommended_items[:k]
    hits = len(set(top_k).intersection(set(relevant_items)))
    return hits / k

In [ ]:
def average_precision_at_k(all_recommendations, all_relevant_items, k):
    user_ids = all_recommendations.keys()  # or range(len(all_recommendations)) if they are lists
    precisions = []

    for user_id in user_ids:
        if user_id not in all_relevant_items:
            continue
        recommended = all_recommendations[user_id]
        relevant    = all_relevant_items[user_id]
        p_at_k = precision_at_k(recommended, relevant, k)
        precisions.append(p_at_k)

    return sum(precisions) / len(precisions)

In [ ]:
breakfast_predictions = {}
lunch_predictions = {}
snack_predictions = {}
dinner_predictions = {}
for prediction in predictions:
    user_id = prediction.uid
    recipe_id = prediction.iid
    rating = prediction.est

    if recipe_id in breakfast_recipe_ids_set:
        if user_id not in breakfast_predictions:
            breakfast_predictions[user_id] = []
        breakfast_predictions[user_id].append((recipe_id, rating))
    if recipe_id in lunch_recipe_ids_set:
        if user_id not in lunch_predictions:
            lunch_predictions[user_id] = []
        lunch_predictions[user_id].append((recipe_id, rating))
    if recipe_id in snack_recipe_ids_set:
        if user_id not in snack_predictions:
            snack_predictions[user_id] = []
        snack_predictions[user_id].append((recipe_id, rating))
    if recipe_id in dinner_recipe_ids_set:
        if user_id not in dinner_predictions:
            dinner_predictions[user_id] = []
        dinner_predictions[user_id].append((recipe_id, rating))


In [ ]:
breakfast_predictions_k = {}
lunch_predictions_k = {}
snack_predictions_k = {}
dinner_predictions_k = {}

breakfast_predictions_k_full = {}

k = 10

for user_id, ratings in breakfast_predictions.items():
    ratings.sort(key=lambda x: x[1], reverse=True)
    breakfast_predictions_k[user_id] = [recipe_id for recipe_id, _ in ratings[:(2*k)]]
    breakfast_predictions_k_full[user_id] = ratings[:(2*k)]

for user_id, ratings in lunch_predictions.items():
    ratings.sort(key=lambda x: x[1], reverse=True)
    lunch_predictions_k[user_id] = [recipe_id for recipe_id, _ in ratings[:(2*k)]]

for user_id, ratings in snack_predictions.items():
    ratings.sort(key=lambda x: x[1], reverse=True)
    snack_predictions_k[user_id] = [recipe_id for recipe_id, _ in ratings[:(2*k)]]

for user_id, ratings in dinner_predictions.items():
    ratings.sort(key=lambda x: x[1], reverse=True)
    dinner_predictions_k[user_id] = [recipe_id for recipe_id, _ in ratings[:(2*k)]]

print(len(breakfast_predictions_k))
print(len(lunch_predictions_k))
print(len(snack_predictions_k))
print(len(dinner_predictions_k))
breakfast_predictions_k_relevant = {}
lunch_predictions_k_relevant = {}
snack_predictions_k_relevant = {}
dinner_predictions_k_relevant = {}

for user_id, predictions in breakfast_predictions_k.items():
    if (len(predictions) > k):
        breakfast_predictions_k_relevant[user_id] = predictions

for user_id, predictions in lunch_predictions_k.items():
    if (len(predictions) > k):
        lunch_predictions_k_relevant[user_id] = predictions

for user_id, predictions in snack_predictions_k.items():
    if (len(predictions) > k):
        snack_predictions_k_relevant[user_id] = predictions

for user_id, predictions in dinner_predictions_k.items():
    if (len(predictions) > k):
        dinner_predictions_k_relevant[user_id] = predictions

In [ ]:
breakfast_precision = average_precision_at_k(breakfast_predictions_k_relevant, relevant_breakfast_dict, k)
lunch_precision = average_precision_at_k(lunch_predictions_k_relevant, relevant_lunch_dict, k)
snack_precision = average_precision_at_k(snack_predictions_k_relevant, relevant_snack_dict, k)
dinner_precision = average_precision_at_k(dinner_predictions_k_relevant, relevant_dinner_dict, k)

print(f"Breakfast Precision@10: {breakfast_precision}")
print(f"Lunch Precision@10: {lunch_precision}")
print(f"Snack Precision@10: {snack_precision}")
print(f"Dinner Precision@10: {dinner_precision}")